# 05 - Error Analysis & Linguistic Failure Modes
## Diagnosing False Positives, False Negatives, and Model Blindspots

In this notebook, we perform structured error analysis on the test predictions:
1. **Error Segmentation**: Identifying False Positives (Type I) and False Negatives (Type II).
2. **Review Length vs. Error Rate**: Evaluating whether long reviews cause degradation.
3. **Linguistic Failure Modes**:
   - **Sarcasm / Irony**: Superficial praise masking caustic criticism.
   - **Mixed Sentiment**: Strong positive elements (acting, visuals) overwhelmed by fatal flaws (script, ending).
   - **Sentiment Shift / Contrastive Conjunctions**: Sudden pivot words (*"however"*, *"nonetheless"*, *"but"*).
   - **Complex Negation**: Double negatives and idiom-based negations.
4. **Summary & Recommendations for Transformer Architecture**.


In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Load data and pipeline
df = pd.read_csv("../data/processed/train_reviews_clean.csv")
X = df["clean_review"].fillna("")
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = joblib.load("../models/baseline_pipeline.pkl")
probs = pipeline.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

results = pd.DataFrame({
    "review": X_test,
    "actual": y_test,
    "predicted": preds,
    "prob_positive": probs,
    "length": X_test.apply(len),
    "word_count": X_test.apply(lambda x: len(x.split()))
})

results["error_type"] = "Correct"
results.loc[(results["actual"] == 0) & (results["predicted"] == 1), "error_type"] = "False Positive"
results.loc[(results["actual"] == 1) & (results["predicted"] == 0), "error_type"] = "False Negative"

print("Prediction Breakdown:")
print(results["error_type"].value_counts())


### 1. Error Rate Across Review Length Quantiles


In [ ]:
results["length_bin"] = pd.qcut(results["word_count"], q=5, labels=["Very Short", "Short", "Medium", "Long", "Very Long"])

error_by_bin = results.groupby("length_bin")["error_type"].apply(
    lambda s: (s != "Correct").mean() * 100
).reset_index(name="error_rate_pct")

plt.figure(figsize=(7, 4))
sns.barplot(data=error_by_bin, x="length_bin", y="error_rate_pct", palette="mako")
plt.title("Error Rate (%) by Review Length Category", fontsize=13, fontweight='bold')
plt.xlabel("Review Length Quantile")
plt.ylabel("Error Rate (%)")
plt.ylim(0, 20)
plt.tight_layout()
plt.show()


### 2. High-Confidence Errors (Most Deceptive Reviews)


In [ ]:
# False Positives where model was extremely confident it was positive (> 85% prob)
top_fp = results[results["error_type"] == "False Positive"].sort_values("prob_positive", ascending=False)

print(f"Total False Positives: {len(top_fp)}")
print("=" * 80)
print("TOP FALSE POSITIVE (Negative movie review misclassified as Positive):")
print(f"Predicted Probability of Positive: {top_fp.iloc[0]['prob_positive']*100:.1f}%")
print("\nReview Snippet:")
print(top_fp.iloc[0]["review"][:800] + "...")


In [ ]:
# False Negatives where model was extremely confident it was negative (< 15% prob)
top_fn = results[results["error_type"] == "False Negative"].sort_values("prob_positive", ascending=True)

print(f"Total False Negatives: {len(top_fn)}")
print("=" * 80)
print("TOP FALSE NEGATIVE (Positive movie review misclassified as Negative):")
print(f"Predicted Probability of Positive: {top_fn.iloc[0]['prob_positive']*100:.1f}%")
print("\nReview Snippet:")
print(top_fn.iloc[0]["review"][:800] + "...")


### 3. Key Findings & Motivation for Transformers:
1. **Bag-of-Words Limitation**: The classical baseline counts word frequencies regardless of position. When a reviewer spends 3 paragraphs praising the actors (*"brilliant"*, *"phenomenal"*, *"charming"*) but ends with *"the movie as a whole is utterly unwatchable"*, TF-IDF weights the positive words more heavily.
2. **Sarcasm**: Phrases like *"Oh what a masterclass of cinema"* are taken literally by bag-of-words models.
3. **Contrastive Connectives**: Words like *"however"*, *"but"*, *"yet"* indicate a sentiment shift, which requires attention mechanisms (such as DistilBERT, RoBERTa, and DeBERTa) to model sentence compositionality.
